In [2]:
from pyspark.sql import functions as F

def transform_fraudData_features(input_json: list[dict]=[], lakehoueseTable: str=''):
    """
    Convert JSON input to Spark DataFrame and apply feature engineering.

    Parameters:
    ----------
    spark : SparkSession
    input_json : dict or list[dict]
        Example:
        {
            "step": 84,
            "amount": 1000,
            "oldbalanceOrg": 5000,
            "newbalanceOrig": 4000,
            "oldbalanceDest": 2000,
            "newbalanceDest": 3000,
            "nameOrig": "C123",
            "nameDest": "M456"
        }

    Returns:
    -------
    Spark DataFrame with engineered features
    """

    # ------------------------------------------------------------------
    # 1. Convert JSON → Spark DataFrame or Read from a Lakehouse Table -
    # ------------------------------------------------------------------
    
    if not input_json and not lakehoueseTable:
        raise ValueError('Invalid inputs!')


    if lakehoueseTable:
        df = spark.read.table(lakehoueseTable)
    
    if input_json:
        df = spark.createDataFrame(input_json)

   # Drop leakage columns
    df = df.drop("isFlaggedFraud")

    # ======================================
    # 2. Balance features (VERY IMPORTANT) =
    # ======================================
    df = df.withColumn(
        "orig_balance_diff",
        F.col("oldbalanceOrg") - F.col("newbalanceOrig") - F.col("amount")
    )

    df = df.withColumn(
        "dest_balance_diff",
        F.col("newbalanceDest") - F.col("oldbalanceDest") - F.col("amount")
    )

    # =========================
    # 3. Zero balance indicators
    # =========================
    df = df.withColumn(
        "orig_zero",
        (F.col("oldbalanceOrg") == 0).cast("int")
    )

    df = df.withColumn(
        "dest_zero",
        (F.col("oldbalanceDest") == 0).cast("int")
    )

    # =========================
    # 4. Amount features
    # =========================
    df = df.withColumn(
        "log_amount",
        F.log1p(F.col("amount"))
    )

    df = df.withColumn(
        "amount_to_balance",
        F.col("amount") / (F.col("oldbalanceOrg") + F.lit(1))
    )

    # =========================
    # 5. Error flags (VERY POWERFUL)
    # =========================
    df = df.withColumn(
        "orig_error",
        ((F.col("oldbalanceOrg") - F.col("amount")) != F.col("newbalanceOrig")).cast("int")
    )

    df = df.withColumn(
        "dest_error",
        ((F.col("oldbalanceDest") + F.col("amount")) != F.col("newbalanceDest")).cast("int")
    )

    # =========================
    # 6. Extract entity type from names
    # =========================
    df = df.withColumn(
        "orig_name_type",
        F.upper(F.substring(F.col("nameOrig"), 1, 1))
    )

    df = df.withColumn(
        "dest_name_type",
        F.upper(F.substring(F.col("nameDest").cast('string'), 1, 1))
    )


    # =========================
    # 7. Frequency features (IMPORTANT)
    # =========================
    orig_counts = df.groupBy("nameOrig").count().withColumnRenamed("count", "orig_name_freq")
    dest_counts = df.groupBy("nameDest").count().withColumnRenamed("count", "dest_name_freq")

    df = df.join(orig_counts, on="nameOrig", how="left")
    df = df.join(dest_counts, on="nameDest", how="left")


    # =========================
    # 8. Pair frequency (VERY STRONG FEATURE)
    # =========================
    pair_counts = df.groupBy("nameOrig", "nameDest").count().withColumnRenamed("count", "pair_freq")

    df = df.join(pair_counts, on=["nameOrig", "nameDest"], how="left")


    # -------------------------
    # 9. Balance ratio features
    # -------------------------
    df = df.withColumn(
        "balance_ratio_orig",
        F.col("newbalanceOrig") / (F.col("oldbalanceOrg") + 1)
    )

    df = df.withColumn(
        "balance_ratio_dest",
        F.col("newbalanceDest") / (F.col("oldbalanceDest") + 1)
    )

    # -------------------------
    # 10. Absolute differences
    # -------------------------
    df = df.withColumn("abs_orig_diff", F.abs(F.col("orig_balance_diff")))
    df = df.withColumn("abs_dest_diff", F.abs(F.col("dest_balance_diff")))

    # -------------------------
    # 11. Amount anomaly features
    # -------------------------
    df = df.withColumn(
        "amount_vs_orig",
        F.col("amount") / (F.col("oldbalanceOrg") + 1)
    )

    df = df.withColumn(
        "amount_vs_dest",
        F.col("amount") / (F.col("oldbalanceDest") + 1)
    )

    # -------------------------
    # 12. Zero behavior flags
    # -------------------------
    df = df.withColumn(
        "orig_zero_flag",
        F.when(
            (F.col("oldbalanceOrg") == 0) & (F.col("newbalanceOrig") == 0),
            1
        ).otherwise(0)
    )

    df = df.withColumn(
        "dest_zero_flag",
        F.when(
            (F.col("oldbalanceDest") == 0) & (F.col("newbalanceDest") == 0),
            1
        ).otherwise(0)
    ) 

    # -------------------------
    # 14. Time features
    # -------------------------
    df = df.withColumn("hour", F.col("step") % 24)
    df = df.withColumn(
        "is_night",
        F.when(F.col("hour").isin([0,1,2,3,4]), 1).otherwise(0)
    )

    # -------------------------
    # 15. Non-linear transforms
    # -------------------------
    df = df.withColumn("sqrt_amount", F.sqrt(F.col("amount")))
    df = df.withColumn("log_balance_orig", F.log1p(F.col("oldbalanceOrg")))
    df = df.withColumn("log_balance_dest", F.log1p(F.col("oldbalanceDest")))

    # ------------------------------------------------------------
    # ⚠️ IMPORTANT NOTE ABOUT FREQUENCY FEATURES
    # ------------------------------------------------------------
    # These require full dataset (groupBy):
    # - orig_name_freq
    # - dest_name_freq
    # - pair_freq
    #
    # ❌ Not suitable for single JSON input
    # ✅ Should be precomputed and joined from a lookup table

    return df


StatementMeta(, 1eaa3380-c6d6-4928-9e81-008f231e173a, 4, Finished, Available, Finished, False)